# **Transformer Model for Text Classification**

## Text classification
Let's build a text classification model using PyTorch and torchtext to classify news articles into one of the four categories: World, Sports, Business, and Sci/Tech.


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import AG_NEWS

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Import bank dataset**

Load the AG_NEWS dataset for the train split and split it into input text and corresponding labels:


In [2]:
train_iter = AG_NEWS(split = "train")
train_iter

ShardingFilterIterDataPipe

In [3]:
label, text = next(iter(train_iter))
print(f"label:{label}, text: {text}")

label:3, text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [4]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
ag_news_label[label]

'Business'

Finding number of class in the dataset

In [5]:
num_classes = len(set([label for (label, text) in train_iter]))
print(f"number of classes: {num_classes}")

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\datapipes\iter\combining.py:297: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


number of classes: 4


### **Build Vocabulary**

In [6]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens (dataset):
    for (label,text) in dataset:
        yield tokenizer(text)

In [7]:
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [8]:
vocab["wall"]

431

#### Build Pipeline and Label Pipeline to Get Token Indices From Vocabulary

In [9]:
def text_pipeline (input_text):
    return vocab(tokenizer(input_text))

In [10]:
sample_input = "Bears Claw Back Into the Black (Reuters) Reuters"
text_pipeline(sample_input)

[1605, 14838, 113, 66, 2, 848, 13, 27, 14, 27]

In [72]:
def label_pipeline(x):
    return int(x)

## **Dataset Initializatioin**

In [12]:
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import random_split

In [13]:
# initializing training and testing iter
train_iter, test_iter = AG_NEWS()

# initializing training and testing dataset
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

num_train = int(len(train_dataset) * 0.95)

train_split, valid_split = random_split(train_dataset,[num_train,(len(train_dataset)-num_train)])

### **Initializing Device**

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

### **Data Loader**


In [15]:
from torchtext.functional import pad_sequence

In [16]:
# we create data loaders using by split datasets

def collate_func (batch):
    
    label_list = []
    text_list = []
    
    for _label, _text in batch:
        label_list.append(label_pipeline(_label)) 
        text_list.append(torch.tensor(text_pipeline(_text),dtype=torch.int64))
    
    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = pad_sequence(text_list,batch_first=True,padding_value=0)
    
    return label_list.to(device), text_list.to(device)

#### Define DataLoaders

In [17]:
from torch.utils.data import DataLoader

In [18]:
batch_size = 64

train_dataloader = DataLoader(train_split,batch_size,
                              shuffle=True, collate_fn=collate_func)

valid_dataloader = DataLoader(valid_split, batch_size, 
                              shuffle=True, collate_fn=collate_func)

test_dataloader = DataLoader(test_dataset, batch_size, 
                             shuffle=True, collate_fn=collate_func)

### **Build the Model**

#### **Positional Encoding**

In [22]:
import math

In [24]:
class PositionalEmbeddings(nn.Module):
    def __init__(self,dropout, num_dims, max_seq_len = 500):
        super().__init__()
        
        #define positions 
        positions = torch.arange(0,max_seq_len,dtype=torch.float).unsqueeze(1)
        
        # define zero matrix
        pe = torch.zeros(max_seq_len,num_dims)
        
        div_term = torch.exp(
            torch.arange(0,num_dims,2).float() *
                (-math.log(10000.0)/ num_dims)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(0)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer("pe",pe)
        
    def forward(self, input_embedding):
        seq_len = input_embedding.size(1) #[batch_size, seq_len, num_dims]
        positional_embeddings = input_embedding + self.pe[:,0:seq_len,:]
        
        return positional_embeddings
          

In [25]:
class ClassificationModel (nn.Module):
    
    def __init__(self, vocab_size, num_embeds, dropout,
                 num_heads,dim_feedforward, num_layers, num_class):
        super(ClassificationModel, self).__init__()
        
        #initiate embeddings layer
        self.embeddings = nn.Embedding(vocab_size, num_embeds)
        
        #initiate positional encoding layer
        self.positional_emeddings = PositionalEmbeddings(dropout,num_embeds)
        
        #define transformerencoderlayer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=num_embeds, 
                                                        nhead=num_heads,
                                                        dim_feedforward=dim_feedforward,
                                                        dropout=dropout)
        
        #define transformer layer
        self.transformer_layer = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=num_layers)
        #define feed forward network
        self.fc_layer = nn.Linear(num_embeds,num_class) 
        self.d_model = num_embeds
        
    def forward(self,input_seq):
        #seq -> [batch_size, seq_len,dims]
        embed_out = self.embeddings(input_seq) * math.sqrt(self.d_model)
        pos_out = self.positional_emeddings(embed_out)
        encoder_layer_out = self.encoder_layer(pos_out)
        transformer_out = self.transformer_layer(encoder_layer_out)
        transformer_out = transformer_out.mean(dim = 1)
        
        out = self.fc_layer(transformer_out)
        
        return out

#### Create a Model

In [28]:
vocab_size = len(vocab)
n_dims = 4
n_heads = 2
n_layers = 2
dropout = 0.1
dim_feedforward = 2

In [29]:
classify_model = ClassificationModel(vocab_size,n_dims,
                                     dropout,n_heads,
                                     dim_feedforward,n_layers,num_classes)

### **Model Evalute**

In [50]:
def model_evalute(model, valid_dataloader):
    acc_val, count = 0,0
    
    with torch.no_grad():
        for idx,(label,text) in enumerate(valid_dataloader):
            model.eval()
            predict_label = model(text)
                
            acc_val += (predict_label.argmax(1) == label).sum().item()
            count += label.size(0)
    
    
    
    return acc_val/count
        
            

### **Train the Model**

In [51]:
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

In [52]:
LR = 0.3
criterion = CrossEntropyLoss()
optimizer = SGD(classify_model.parameters(),lr = LR)
schduler = StepLR(optimizer,step_size=1,gamma=0.1)


In [55]:
def model_train(model, train_dataloader,criterion,optimizer, epochs = 3):
    
    epoch_loss = []
    accuracy_epoch = []
    accuracy_old = 0
    
    for epoch in tqdm(range(0, epochs)):
        model.train()
        cum_loss = 0
        
        for id,(label, text) in enumerate(train_dataloader):
            
            optimizer.zero_grad()
            label = label.to(device)
            text = text.to(device)
            
            #model prediction
            predict_label = model(text)
            loss = criterion(predict_label,label)
            #loss backward
            loss.backward()
            #limit gradient for explording
            torch.nn.utils.clip_grad_norm_(model.parameters(),0.1)
            
            #increase optimizer
            optimizer.step()
            
            cum_loss += loss.item()
        average_loss = cum_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}, Loss: {average_loss:.4f}")
        
        epoch_loss.append(average_loss)
        accuacy_val = model_evalute(model,valid_dataloader)
        accuracy_epoch.append(accuacy_val)
        
        if accuracy_old < accuacy_val:
            accuracy_old = accuacy_val
            torch.save(model.state_dict(), "classification_model.pth")
        
    return accuracy_epoch, epoch_loss
        

Model Training

In [56]:
accuracy_list, loss_list = model_train(classify_model,
                                       train_dataloader,criterion,
                                       optimizer)

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1, Loss: 1.1910


 33%|███▎      | 1/3 [02:22<04:44, 142.07s/it]

Epoch 2, Loss: 1.1728


 67%|██████▋   | 2/3 [04:54<02:28, 148.09s/it]

Epoch 3, Loss: 1.1566


100%|██████████| 3/3 [07:35<00:00, 151.74s/it]


In [61]:
sample_text = "Sri lanka is a beautiful place"
tokens = text_pipeline(sample_text)
tokens = torch.tensor(tokens).unsqueeze(0)
tokens.shape


torch.Size([1, 6])

In [74]:
sample_text = "sri lanka economic is bad"

tokens = text_pipeline(sample_text)
tokens = torch.tensor(tokens).unsqueeze(0)

classify_model.eval()

with torch.no_grad():
    output = classify_model(tokens)
    label_predict = output.argmax(dim=1)

predicted_label = label_pipeline(label_predict.item())

ag_news_label[predicted_label+1]

'Business'